# ✈️ Airline Delay Dataset — Data Cleaning Pipeline

**Dataset:** [Airline Delay Causes — Kaggle](https://www.kaggle.com/datasets/giovamata/airlinedelaycauses/data)  
**Source:** U.S. Bureau of Transportation Statistics (BTS)

---

## 📋 Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | Import Libraries |
| 2 | Load & Initial Exploration |
| 3 | Segment Flights (Cancelled / Diverted / Delayed) |
| 4 | Drop Unnecessary Columns |
| 5 | Rename Columns |
| 6 | Fix Data Types — Date & Time |
| 7 | Handle Missing Values |
| 8 | Remove Duplicates |
| 9 | Outlier Detection & Capping |
| 10 | Memory Optimization (Category Types) |
| 11 | Feature Engineering & Validation |
| 12 | Reorder Columns & Export |

In [1]:
import pandas as pd
import numpy as np

In [14]:
df=pd.read_csv('DelayedFlights_Source.csv')
pd.set_option('display.max_columns', None)
df.head(10)

,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,0,N,0,NaN,NaN,NaN,NaN,NaN
1,1,2008,1,3,4,754.0,735,1002.0,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
2,2,2008,1,3,4,628.0,620,804.0,750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,0,N,0,NaN,NaN,NaN,NaN,NaN
3,4,2008,1,3,4,1829.0,1755,1959.0,1925,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,0,N,0,2.0,0.0,0.0,0.0,32.0
4,5,2008,1,3,4,1940.0,1915,2121.0,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
5,6,2008,1,3,4,1937.0,1830,2037.0,1940,WN,509,N763SW,240.0,250.0,230.0,57.0,67.0,IND,LAS,1591,3.0,7.0,0,N,0,10.0,0.0,0.0,0.0,47.0
6,10,2008,1,3,4,706.0,700,916.0,915,WN,100,N690SW,130.0,135.0,106.0,1.0,6.0,IND,MCO,828,5.0,19.0,0,N,0,NaN,NaN,NaN,NaN,NaN
7,11,2008,1,3,4,1644.0,1510,1845.0,1725,WN,1333,N334SW,121.0,135.0,107.0,80.0,94.0,IND,MCO,828,6.0,8.0,0,N,0,8.0,0.0,0.0,0.0,72.0
8,15,2008,1,3,4,1029.0,1020,1021.0,1010,WN,2272,N263WN,52.0,50.0,37.0,11.0,9.0,IND,MDW,162,6.0,9.0,0,N,0,NaN,NaN,NaN,NaN,NaN
9,16,2008,1,3,4,1452.0,1425,1640.0,1625,WN,675,N286WN,228.0,240.0,213.0,15.0,27.0,IND,PHX,1489,7.0,8.0,0,N,0,3.0,0.0,0.0,0.0,12.0


In [15]:
del df['Unnamed: 0']
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1936758 entries, 0 to 1936757
Data columns (total 29 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Year               int64  
 1   Month              int64  
 2   DayofMonth         int64  
 3   DayOfWeek          int64  
 4   DepTime            float64
 5   CRSDepTime         int64  
 6   ArrTime            float64
 7   CRSArrTime         int64  
 8   UniqueCarrier      object 
 9   FlightNum          int64  
 10  TailNum            object 
 11  ActualElapsedTime  float64
 12  CRSElapsedTime     float64
 13  AirTime            float64
 14  ArrDelay           float64
 15  DepDelay           float64
 16  Origin             object 
 17  Dest               object 
 18  Distance           int64  
 19  TaxiIn             float64
 20  TaxiOut            float64
 21  Cancelled          int64  
 22  CancellationCode   object 
 23  Diverted           int64  
 24  CarrierDelay       float64
 25  WeatherDelay      

In [16]:
df.isnull().sum()

Year                      0
Month                     0
DayofMonth                0
DayOfWeek                 0
DepTime                   0
CRSDepTime                0
ArrTime                7110
CRSArrTime                0
UniqueCarrier             0
FlightNum                 0
TailNum                   5
ActualElapsedTime      8387
CRSElapsedTime          198
AirTime                8387
ArrDelay               8387
DepDelay                  0
Origin                    0
Dest                      0
Distance                  0
TaxiIn                 7110
TaxiOut                 455
Cancelled                 0
CancellationCode          0
Diverted                  0
CarrierDelay         689270
WeatherDelay         689270
NASDelay             689270
SecurityDelay        689270
LateAircraftDelay    689270
dtype: int64

In [20]:
# Get the number of Cancelled Flights Vs Delayed Flights
df['Cancelled'].value_counts()

Cancelled
0    1936125
1        633
Name: count, dtype: int64

In [18]:
# Extracting Cancelled Flights
Cancelled_Flights =df[df['Cancelled'] == 1]
Cancelled_Flights.to_csv('Cancelled_Flights.csv', index=False)
Cancelled_Flights.head(10)

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
1542406,2008,10,27,1,1622.0,1420,NaN,1520,WN,27,N601WN,NaN,60.0,NaN,NaN,122.0,HOU,HRL,276,NaN,19.0,1,A,0,NaN,NaN,NaN,NaN,NaN
1546593,2008,10,25,6,1323.0,1255,NaN,1442,XE,2347,N26549,NaN,107.0,NaN,NaN,28.0,CLT,EWR,529,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN
1547161,2008,10,22,3,1825.0,1815,NaN,1927,XE,2819,N12946,NaN,72.0,NaN,NaN,10.0,JAN,IAH,351,NaN,NaN,1,C,0,NaN,NaN,NaN,NaN,NaN
1547178,2008,10,22,3,1733.0,1715,NaN,1818,XE,2890,N16944,NaN,63.0,NaN,NaN,18.0,IAH,BTR,253,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN
1548271,2008,10,15,3,1943.0,1745,NaN,1857,XE,2117,N26545,NaN,72.0,NaN,NaN,118.0,IAH,HRL,295,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN
1548430,2008,10,15,3,1610.0,1600,NaN,1738,XE,2920,N14558,NaN,98.0,NaN,NaN,10.0,IAH,MEM,469,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN
1550787,2008,10,5,7,1711.0,1653,NaN,1821,YV,7148,N570ML,NaN,88.0,NaN,NaN,18.0,IAD,TYS,419,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN
1551498,2008,10,10,5,1502.0,1450,NaN,1546,YV,7097,N455YV,NaN,56.0,NaN,NaN,12.0,DEN,ASE,125,NaN,14.0,1,B,0,NaN,NaN,NaN,NaN,NaN
1554099,2008,10,28,2,1735.0,1410,NaN,1529,YV,7262,N571ML,NaN,79.0,NaN,NaN,205.0,SYR,IAD,296,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN
1554263,2008,10,30,4,2221.0,1848,NaN,2112,YV,7281,N514MJ,NaN,204.0,NaN,NaN,213.0,IAD,AUS,1297,NaN,12.0,1,A,0,NaN,NaN,NaN,NaN,NaN


In [22]:
# finding the number of diverted flights
df['Diverted'].value_counts()

# Extracting Diverted Flights
Diverted_Flights =df[df['Diverted'] == 1]
Diverted_Flights.to_csv('Diverted_Flights.csv', index=False)
Diverted_Flights.head(10)

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
1280,2008,1,3,4,922.0,915,NaN,1050,WN,1069,N630WN,NaN,95.0,NaN,NaN,7.0,SAN,SMF,480,NaN,12.0,0,N,1,NaN,NaN,NaN,NaN,NaN
1372,2008,1,3,4,2325.0,1900,NaN,2030,WN,2092,N302SW,NaN,90.0,NaN,NaN,265.0,SFO,SAN,447,NaN,11.0,0,N,1,NaN,NaN,NaN,NaN,NaN
1776,2008,1,4,5,1949.0,1905,NaN,1910,WN,1403,N504SW,NaN,65.0,NaN,NaN,44.0,BOI,RNO,335,NaN,11.0,0,N,1,NaN,NaN,NaN,NaN,NaN
1831,2008,1,4,5,737.0,705,NaN,825,WN,178,N718SW,NaN,80.0,NaN,NaN,32.0,BUR,SMF,358,NaN,13.0,0,N,1,NaN,NaN,NaN,NaN,NaN
2244,2008,1,4,5,1849.0,1630,NaN,1755,WN,239,N636WN,NaN,85.0,NaN,NaN,139.0,LAS,RNO,345,NaN,12.0,0,N,1,NaN,NaN,NaN,NaN,NaN
2245,2008,1,4,5,1905.0,1855,NaN,2015,WN,298,N728SW,NaN,80.0,NaN,NaN,10.0,LAS,RNO,345,NaN,12.0,0,N,1,NaN,NaN,NaN,NaN,NaN
2720,2008,1,4,5,1721.0,1605,NaN,1655,WN,985,N613SW,NaN,50.0,NaN,NaN,76.0,OAK,RNO,180,NaN,9.0,0,N,1,NaN,NaN,NaN,NaN,NaN
2831,2008,1,4,5,1837.0,1420,NaN,1540,WN,3513,N348SW,NaN,80.0,NaN,NaN,257.0,PDX,RNO,444,NaN,6.0,0,N,1,NaN,NaN,NaN,NaN,NaN
3075,2008,1,4,5,1832.0,1755,NaN,1935,WN,1511,N259WN,NaN,100.0,NaN,NaN,37.0,SAN,RNO,488,NaN,8.0,0,N,1,NaN,NaN,NaN,NaN,NaN
3179,2008,1,4,5,2226.0,2040,NaN,2150,WN,2285,N368SW,NaN,70.0,NaN,NaN,106.0,SJC,LAX,308,NaN,17.0,0,N,1,NaN,NaN,NaN,NaN,NaN


In [23]:
# Extracting Delayed Flights for Acuarate Analysis

Delayed_Flights = df[(df['Cancelled'] == 0)&(df['Diverted'] == 0)]
Delayed_Flights.to_csv('Delayed_Flights.csv', index=False)
Delayed_Flights.head(10)


,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,0,N,0,NaN,NaN,NaN,NaN,NaN
1,2008,1,3,4,754.0,735,1002.0,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
2,2008,1,3,4,628.0,620,804.0,750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,0,N,0,NaN,NaN,NaN,NaN,NaN
3,2008,1,3,4,1829.0,1755,1959.0,1925,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,0,N,0,2.0,0.0,0.0,0.0,32.0
4,2008,1,3,4,1940.0,1915,2121.0,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
5,2008,1,3,4,1937.0,1830,2037.0,1940,WN,509,N763SW,240.0,250.0,230.0,57.0,67.0,IND,LAS,1591,3.0,7.0,0,N,0,10.0,0.0,0.0,0.0,47.0
6,2008,1,3,4,706.0,700,916.0,915,WN,100,N690SW,130.0,135.0,106.0,1.0,6.0,IND,MCO,828,5.0,19.0,0,N,0,NaN,NaN,NaN,NaN,NaN
7,2008,1,3,4,1644.0,1510,1845.0,1725,WN,1333,N334SW,121.0,135.0,107.0,80.0,94.0,IND,MCO,828,6.0,8.0,0,N,0,8.0,0.0,0.0,0.0,72.0
8,2008,1,3,4,1029.0,1020,1021.0,1010,WN,2272,N263WN,52.0,50.0,37.0,11.0,9.0,IND,MDW,162,6.0,9.0,0,N,0,NaN,NaN,NaN,NaN,NaN
9,2008,1,3,4,1452.0,1425,1640.0,1625,WN,675,N286WN,228.0,240.0,213.0,15.0,27.0,IND,PHX,1489,7.0,8.0,0,N,0,3.0,0.0,0.0,0.0,12.0


In [1]:
import pandas as pd 

Delayed_Flights=pd.read_csv(r"E:\.Eng_Ahmed\Data Engineer\.ITI\power bi\last\Cleaned_delayed_flights.csv")
Delayed_Flights

,Date,year,month,day,DayOfWeek,UniqueCarrier,FlightNum,TailNum,Origin,Dest,...,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,Delay_sum,ActualElapsedTime,Expected_ElapsedTime,AirTime,TaxiIn,TaxiOut
0,1/3/2008,2008,1,3,4,WN,335,N712SW,IAD,TPA,...,0,0,0,0,0,128,150,116,4,8
1,1/3/2008,2008,1,3,4,WN,3231,N772SW,IAD,TPA,...,0,0,0,0,0,128,145,113,5,10
2,1/3/2008,2008,1,3,4,WN,448,N428WN,IND,BWI,...,0,0,0,0,0,96,90,76,3,17
3,1/3/2008,2008,1,3,4,WN,3920,N464WN,IND,BWI,...,0,0,0,32,34,90,90,77,3,10
4,1/3/2008,2008,1,3,4,WN,378,N726SW,IND,JAX,...,0,0,0,0,0,101,115,87,4,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048570,6/12/2008,2008,6,12,4,AA,1865,N578AA,DFW,BUR,...,0,12,0,0,24,197,185,179,3,15
1048571,6/13/2008,2008,6,13,5,AA,1865,N465AA,DFW,BUR,...,0,5,0,35,41,190,185,176,4,10
1048572,6/14/2008,2008,6,14,6,AA,1865,N207AA,DFW,BUR,...,0,9,0,37,46,194,185,166,4,24
1048573,6/15/2008,2008,6,15,7,AA,1865,N455AA,DFW,BUR,...,0,32,0,76,115,217,185,173,3,41


In [3]:
Delayed_Flights['UniqueCarrier'].value_counts()

UniqueCarrier
WN    213885
AA    106552
MQ     82135
UA     81778
OO     73388
XE     62246
US     53684
DL     48692
NW     48265
CO     44152
EV     42782
FL     37022
YV     34760
9E     31677
OH     29046
B6     22771
AS     16428
F9     15975
HA      2593
AQ       744
Name: count, dtype: int64

In [8]:
pd.reset_option('display.max_rows')
Delayed_Flights['TailNum'].value_counts()

TailNum
N325SW    592
N688SW    574
N313SW    567
N641SW    552
N682SW    550
         ... 
N199UA      1
N859NW      1
N77006      1
N74007      1
N27015      1
Name: count, Length: 5042, dtype: int64

In [9]:
Delayed_Flights['Origin'].value_counts()

Origin
ORD    72109
ATL    63704
DFW    51684
DEN    39826
LAX    33386
       ...  
BLI        4
WYS        4
BJI        2
TUP        1
INL        1
Name: count, Length: 296, dtype: int64

In [3]:
# Exploring the data of Delayed Flights
Delayed_Flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1928371 entries, 0 to 1928370
Data columns (total 29 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Year               int64  
 1   Month              int64  
 2   DayofMonth         int64  
 3   DayOfWeek          int64  
 4   DepTime            float64
 5   CRSDepTime         int64  
 6   ArrTime            float64
 7   CRSArrTime         int64  
 8   UniqueCarrier      object 
 9   FlightNum          int64  
 10  TailNum            object 
 11  ActualElapsedTime  float64
 12  CRSElapsedTime     float64
 13  AirTime            float64
 14  ArrDelay           float64
 15  DepDelay           float64
 16  Origin             object 
 17  Dest               object 
 18  Distance           int64  
 19  TaxiIn             float64
 20  TaxiOut            float64
 21  Cancelled          int64  
 22  CancellationCode   object 
 23  Diverted           int64  
 24  CarrierDelay       float64
 25  WeatherDelay      

In [4]:
# Checking for missing values in Delayed Flights dataset
Delayed_Flights.isnull().sum()

Year                      0
Month                     0
DayofMonth                0
DayOfWeek                 0
DepTime                   0
CRSDepTime                0
ArrTime                   0
CRSArrTime                0
UniqueCarrier             0
FlightNum                 0
TailNum                   3
ActualElapsedTime         0
CRSElapsedTime            0
AirTime                   0
ArrDelay                  0
DepDelay                  0
Origin                    0
Dest                      0
Distance                  0
TaxiIn                    0
TaxiOut                   0
Cancelled                 0
CancellationCode          0
Diverted                  0
CarrierDelay         680883
WeatherDelay         680883
NASDelay             680883
SecurityDelay        680883
LateAircraftDelay    680883
dtype: int64

In [5]:
pd.set_option('display.max_columns', None)
Delayed_Flights[Delayed_Flights['CarrierDelay'].isnull()].head(10)

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,0,N,0,NaN,NaN,NaN,NaN,NaN
1,2008,1,3,4,754.0,735,1002.0,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
2,2008,1,3,4,628.0,620,804.0,750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,0,N,0,NaN,NaN,NaN,NaN,NaN
4,2008,1,3,4,1940.0,1915,2121.0,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
6,2008,1,3,4,706.0,700,916.0,915,WN,100,N690SW,130.0,135.0,106.0,1.0,6.0,IND,MCO,828,5.0,19.0,0,N,0,NaN,NaN,NaN,NaN,NaN
8,2008,1,3,4,1029.0,1020,1021.0,1010,WN,2272,N263WN,52.0,50.0,37.0,11.0,9.0,IND,MDW,162,6.0,9.0,0,N,0,NaN,NaN,NaN,NaN,NaN
10,2008,1,3,4,754.0,745,940.0,955,WN,1144,N778SW,226.0,250.0,205.0,-15.0,9.0,IND,PHX,1489,5.0,16.0,0,N,0,NaN,NaN,NaN,NaN,NaN
14,2008,1,3,4,1900.0,1840,1956.0,1950,WN,717,N786SW,56.0,70.0,49.0,6.0,20.0,ISP,BWI,220,2.0,5.0,0,N,0,NaN,NaN,NaN,NaN,NaN
15,2008,1,3,4,1039.0,1030,1133.0,1140,WN,1244,N714CB,54.0,70.0,47.0,-7.0,9.0,ISP,BWI,220,2.0,5.0,0,N,0,NaN,NaN,NaN,NaN,NaN
16,2008,1,3,4,1520.0,1455,1619.0,1605,WN,2553,N394SW,59.0,70.0,50.0,14.0,25.0,ISP,BWI,220,2.0,7.0,0,N,0,NaN,NaN,NaN,NaN,NaN


In [6]:
# DROPING UNNECESSARY COLUMNS

del Delayed_Flights['Cancelled']
del Delayed_Flights['Diverted']
del Delayed_Flights['CancellationCode']


In [7]:
Delayed_Flights.head(10)

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,NaN,NaN,NaN,NaN,NaN
1,2008,1,3,4,754.0,735,1002.0,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,NaN,NaN,NaN,NaN,NaN
2,2008,1,3,4,628.0,620,804.0,750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,NaN,NaN,NaN,NaN,NaN
3,2008,1,3,4,1829.0,1755,1959.0,1925,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,2.0,0.0,0.0,0.0,32.0
4,2008,1,3,4,1940.0,1915,2121.0,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,NaN,NaN,NaN,NaN,NaN
5,2008,1,3,4,1937.0,1830,2037.0,1940,WN,509,N763SW,240.0,250.0,230.0,57.0,67.0,IND,LAS,1591,3.0,7.0,10.0,0.0,0.0,0.0,47.0
6,2008,1,3,4,706.0,700,916.0,915,WN,100,N690SW,130.0,135.0,106.0,1.0,6.0,IND,MCO,828,5.0,19.0,NaN,NaN,NaN,NaN,NaN
7,2008,1,3,4,1644.0,1510,1845.0,1725,WN,1333,N334SW,121.0,135.0,107.0,80.0,94.0,IND,MCO,828,6.0,8.0,8.0,0.0,0.0,0.0,72.0
8,2008,1,3,4,1029.0,1020,1021.0,1010,WN,2272,N263WN,52.0,50.0,37.0,11.0,9.0,IND,MDW,162,6.0,9.0,NaN,NaN,NaN,NaN,NaN
9,2008,1,3,4,1452.0,1425,1640.0,1625,WN,675,N286WN,228.0,240.0,213.0,15.0,27.0,IND,PHX,1489,7.0,8.0,3.0,0.0,0.0,0.0,12.0


In [8]:
#Renaming columns for better understanding
Delayed_Flights.rename(columns={'Year': 'year','Month': 'month','DayofMonth': 'day','DepTime': 'Actual_DepTime','CRSDepTime': 'Scheduled_DepTime','ArrTime':'Actual_ArrTime','CRSArrTime':'Scheduled_ArrTime','CRSElapsedTime':'Expected_ElapsedTime','ArrDelay':'Arrival_Delay','DepDelay':'Departure_Delay',}, inplace=True)
Delayed_Flights.head(10)

,year,month,day,DayOfWeek,Actual_DepTime,Scheduled_DepTime,Actual_ArrTime,Scheduled_ArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,Expected_ElapsedTime,AirTime,Arrival_Delay,Departure_Delay,Origin,Dest,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,NaN,NaN,NaN,NaN,NaN
1,2008,1,3,4,754.0,735,1002.0,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,NaN,NaN,NaN,NaN,NaN
2,2008,1,3,4,628.0,620,804.0,750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,NaN,NaN,NaN,NaN,NaN
3,2008,1,3,4,1829.0,1755,1959.0,1925,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,2.0,0.0,0.0,0.0,32.0
4,2008,1,3,4,1940.0,1915,2121.0,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,NaN,NaN,NaN,NaN,NaN
5,2008,1,3,4,1937.0,1830,2037.0,1940,WN,509,N763SW,240.0,250.0,230.0,57.0,67.0,IND,LAS,1591,3.0,7.0,10.0,0.0,0.0,0.0,47.0
6,2008,1,3,4,706.0,700,916.0,915,WN,100,N690SW,130.0,135.0,106.0,1.0,6.0,IND,MCO,828,5.0,19.0,NaN,NaN,NaN,NaN,NaN
7,2008,1,3,4,1644.0,1510,1845.0,1725,WN,1333,N334SW,121.0,135.0,107.0,80.0,94.0,IND,MCO,828,6.0,8.0,8.0,0.0,0.0,0.0,72.0
8,2008,1,3,4,1029.0,1020,1021.0,1010,WN,2272,N263WN,52.0,50.0,37.0,11.0,9.0,IND,MDW,162,6.0,9.0,NaN,NaN,NaN,NaN,NaN
9,2008,1,3,4,1452.0,1425,1640.0,1625,WN,675,N286WN,228.0,240.0,213.0,15.0,27.0,IND,PHX,1489,7.0,8.0,3.0,0.0,0.0,0.0,12.0


In [9]:
# Creating a new column for the date of the flight
Delayed_Flights['Date']=pd.to_datetime(Delayed_Flights[['year', 'month', 'day']])
Delayed_Flights.head(10)

,year,month,day,DayOfWeek,Actual_DepTime,Scheduled_DepTime,Actual_ArrTime,Scheduled_ArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,Expected_ElapsedTime,AirTime,Arrival_Delay,Departure_Delay,Origin,Dest,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,Date
0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
1,2008,1,3,4,754.0,735,1002.0,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
2,2008,1,3,4,628.0,620,804.0,750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
3,2008,1,3,4,1829.0,1755,1959.0,1925,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,2.0,0.0,0.0,0.0,32.0,2008-01-03
4,2008,1,3,4,1940.0,1915,2121.0,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
5,2008,1,3,4,1937.0,1830,2037.0,1940,WN,509,N763SW,240.0,250.0,230.0,57.0,67.0,IND,LAS,1591,3.0,7.0,10.0,0.0,0.0,0.0,47.0,2008-01-03
6,2008,1,3,4,706.0,700,916.0,915,WN,100,N690SW,130.0,135.0,106.0,1.0,6.0,IND,MCO,828,5.0,19.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
7,2008,1,3,4,1644.0,1510,1845.0,1725,WN,1333,N334SW,121.0,135.0,107.0,80.0,94.0,IND,MCO,828,6.0,8.0,8.0,0.0,0.0,0.0,72.0,2008-01-03
8,2008,1,3,4,1029.0,1020,1021.0,1010,WN,2272,N263WN,52.0,50.0,37.0,11.0,9.0,IND,MDW,162,6.0,9.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
9,2008,1,3,4,1452.0,1425,1640.0,1625,WN,675,N286WN,228.0,240.0,213.0,15.0,27.0,IND,PHX,1489,7.0,8.0,3.0,0.0,0.0,0.0,12.0,2008-01-03


In [ ]:
# float 804.0  -->int  804 -->str 0804  --> time 08:04

                            #     2400  --> 23:59
# 00:00
# 24 --> 

# 08:04


In [10]:
# cnverting time columns to datetime format

# change from flaot into int

Delayed_Flights['Actual_DepTime'] = Delayed_Flights['Actual_DepTime'].astype(float).astype(int)
Delayed_Flights['Scheduled_DepTime'] = Delayed_Flights['Scheduled_DepTime'].astype(float).astype(int)
Delayed_Flights['Actual_ArrTime'] = Delayed_Flights['Actual_ArrTime'].astype(float).astype(int)
Delayed_Flights['Scheduled_ArrTime'] = Delayed_Flights['Scheduled_ArrTime'].astype(float).astype(int)

Delayed_Flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1928371 entries, 0 to 1928370
Data columns (total 27 columns):
 #   Column                Dtype         
---  ------                -----         
 0   year                  int64         
 1   month                 int64         
 2   day                   int64         
 3   DayOfWeek             int64         
 4   Actual_DepTime        int64         
 5   Scheduled_DepTime     int64         
 6   Actual_ArrTime        int64         
 7   Scheduled_ArrTime     int64         
 8   UniqueCarrier         object        
 9   FlightNum             int64         
 10  TailNum               object        
 11  ActualElapsedTime     float64       
 12  Expected_ElapsedTime  float64       
 13  AirTime               float64       
 14  Arrival_Delay         float64       
 15  Departure_Delay       float64       
 16  Origin                object        
 17  Dest                  object        
 18  Distance              int64         
 19  

In [12]:
Delayed_Flights['Actual_DepTime'] = Delayed_Flights['Actual_DepTime'].astype(str).str.zfill(4)
Delayed_Flights['Scheduled_DepTime']=Delayed_Flights['Scheduled_DepTime'].astype(str).str.zfill(4)
Delayed_Flights['Actual_ArrTime']=Delayed_Flights['Actual_ArrTime'].astype(str).str.zfill(4)
Delayed_Flights['Scheduled_ArrTime']=Delayed_Flights['Scheduled_ArrTime'].astype(str).str.zfill(4)

Delayed_Flights.info()
Delayed_Flights.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1928371 entries, 0 to 1928370
Data columns (total 27 columns):
 #   Column                Dtype         
---  ------                -----         
 0   year                  int64         
 1   month                 int64         
 2   day                   int64         
 3   DayOfWeek             int64         
 4   Actual_DepTime        object        
 5   Scheduled_DepTime     object        
 6   Actual_ArrTime        object        
 7   Scheduled_ArrTime     object        
 8   UniqueCarrier         object        
 9   FlightNum             int64         
 10  TailNum               object        
 11  ActualElapsedTime     float64       
 12  Expected_ElapsedTime  float64       
 13  AirTime               float64       
 14  Arrival_Delay         float64       
 15  Departure_Delay       float64       
 16  Origin                object        
 17  Dest                  object        
 18  Distance              int64         
 19  

,year,month,day,DayOfWeek,Actual_DepTime,Scheduled_DepTime,Actual_ArrTime,Scheduled_ArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,Expected_ElapsedTime,AirTime,Arrival_Delay,Departure_Delay,Origin,Dest,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,Date
0,2008,1,3,4,2003,1955,2211,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
1,2008,1,3,4,0754,0735,1002,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
2,2008,1,3,4,0628,0620,0804,0750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
3,2008,1,3,4,1829,1755,1959,1925,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,2.0,0.0,0.0,0.0,32.0,2008-01-03
4,2008,1,3,4,1940,1915,2121,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,NaN,NaN,NaN,NaN,NaN,2008-01-03


In [13]:
# Replace invalid value (24:00) with 00:00 

Delayed_Flights['Scheduled_DepTime']=Delayed_Flights['Scheduled_DepTime'].replace('2400','0000')
Delayed_Flights['Actual_DepTime']=Delayed_Flights['Actual_DepTime'].replace('2400','0000')
Delayed_Flights['Scheduled_ArrTime']=Delayed_Flights['Scheduled_ArrTime'].replace('2400','0000')
Delayed_Flights['Actual_ArrTime']=Delayed_Flights['Actual_ArrTime'].replace('2400','0000')


In [14]:
Delayed_Flights['Scheduled_DepTime'] = pd.to_datetime(Delayed_Flights['Scheduled_DepTime'],format='%H%M',errors='coerce').dt.time
Delayed_Flights['Actual_DepTime'] = pd.to_datetime(Delayed_Flights['Actual_DepTime'],format='%H%M',errors='coerce').dt.time
Delayed_Flights['Scheduled_ArrTime'] = pd.to_datetime(Delayed_Flights['Scheduled_ArrTime'],format='%H%M',errors='coerce').dt.time
Delayed_Flights['Actual_ArrTime'] = pd.to_datetime(Delayed_Flights['Actual_ArrTime'],format='%H%M',errors='coerce').dt.time

In [15]:
Delayed_Flights.isnull().sum()

year                         0
month                        0
day                          0
DayOfWeek                    0
Actual_DepTime               0
Scheduled_DepTime            0
Actual_ArrTime               0
Scheduled_ArrTime            0
UniqueCarrier                0
FlightNum                    0
TailNum                      3
ActualElapsedTime            0
Expected_ElapsedTime         0
AirTime                      0
Arrival_Delay                0
Departure_Delay              0
Origin                       0
Dest                         0
Distance                     0
TaxiIn                       0
TaxiOut                      0
CarrierDelay            680883
WeatherDelay            680883
NASDelay                680883
SecurityDelay           680883
LateAircraftDelay       680883
Date                         0
dtype: int64

In [16]:
Delayed_Flights

,year,month,day,DayOfWeek,Actual_DepTime,Scheduled_DepTime,Actual_ArrTime,Scheduled_ArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,Expected_ElapsedTime,AirTime,Arrival_Delay,Departure_Delay,Origin,Dest,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,Date
0,2008,1,3,4,20:03:00,19:55:00,22:11:00,22:25:00,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
1,2008,1,3,4,07:54:00,07:35:00,10:02:00,10:00:00,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
2,2008,1,3,4,06:28:00,06:20:00,08:04:00,07:50:00,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
3,2008,1,3,4,18:29:00,17:55:00,19:59:00,19:25:00,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,2.0,0.0,0.0,0.0,32.0,2008-01-03
4,2008,1,3,4,19:40:00,19:15:00,21:21:00,21:10:00,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,NaN,NaN,NaN,NaN,NaN,2008-01-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928366,2008,12,13,6,12:50:00,12:20:00,16:17:00,15:52:00,DL,1621,N938DL,147.0,152.0,120.0,25.0,30.0,MSP,ATL,906,9.0,18.0,3.0,0.0,0.0,0.0,22.0,2008-12-13
1928367,2008,12,13,6,06:57:00,06:00:00,09:04:00,07:49:00,DL,1631,N3743H,127.0,109.0,78.0,75.0,57.0,RIC,ATL,481,15.0,34.0,0.0,57.0,18.0,0.0,0.0,2008-12-13
1928368,2008,12,13,6,10:07:00,08:47:00,11:49:00,10:10:00,DL,1631,N909DA,162.0,143.0,122.0,99.0,80.0,ATL,IAH,689,8.0,32.0,1.0,0.0,19.0,0.0,79.0,2008-12-13
1928369,2008,12,13,6,12:51:00,12:40:00,14:46:00,14:37:00,DL,1639,N646DL,115.0,117.0,89.0,9.0,11.0,IAD,ATL,533,13.0,13.0,NaN,NaN,NaN,NaN,NaN,2008-12-13


In [17]:
# Handlling Null Values 

Delayed_Flights.fillna({'CarrierDelay': 0, 'WeatherDelay': 0, 'NASDelay': 0, 'SecurityDelay': 0, 'LateAircraftDelay': 0}, inplace=True)
Delayed_Flights.dropna(subset=['TailNum'], inplace=True)
Delayed_Flights.isnull().sum()

year                    0
month                   0
day                     0
DayOfWeek               0
Actual_DepTime          0
Scheduled_DepTime       0
Actual_ArrTime          0
Scheduled_ArrTime       0
UniqueCarrier           0
FlightNum               0
TailNum                 0
ActualElapsedTime       0
Expected_ElapsedTime    0
AirTime                 0
Arrival_Delay           0
Departure_Delay         0
Origin                  0
Dest                    0
Distance                0
TaxiIn                  0
TaxiOut                 0
CarrierDelay            0
WeatherDelay            0
NASDelay                0
SecurityDelay           0
LateAircraftDelay       0
Date                    0
dtype: int64

In [18]:
Delayed_Flights['Delay_sum'] = (
    Delayed_Flights['CarrierDelay'] +
    Delayed_Flights['WeatherDelay'] +
    Delayed_Flights['NASDelay'] +
    Delayed_Flights['SecurityDelay'] +
    Delayed_Flights['LateAircraftDelay']
)

In [20]:
Delayed_Flights.shape

(1928368, 28)

In [ ]:
Delayed_Flights.drop_duplicates(inplace=True)
Delayed_Flights.info()

(1928366, 28)

In [23]:
Delayed_Flights

,year,month,day,DayOfWeek,Actual_DepTime,Scheduled_DepTime,Actual_ArrTime,Scheduled_ArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,Expected_ElapsedTime,AirTime,Arrival_Delay,Departure_Delay,Origin,Dest,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,Date,Delay_sum
0,2008,1,3,4,20:03:00,19:55:00,22:11:00,22:25:00,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,0.0,0.0,0.0,0.0,0.0,2008-01-03,0.0
1,2008,1,3,4,07:54:00,07:35:00,10:02:00,10:00:00,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,0.0,0.0,0.0,0.0,0.0,2008-01-03,0.0
2,2008,1,3,4,06:28:00,06:20:00,08:04:00,07:50:00,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,0.0,0.0,0.0,0.0,0.0,2008-01-03,0.0
3,2008,1,3,4,18:29:00,17:55:00,19:59:00,19:25:00,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,2.0,0.0,0.0,0.0,32.0,2008-01-03,34.0
4,2008,1,3,4,19:40:00,19:15:00,21:21:00,21:10:00,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,0.0,0.0,0.0,0.0,0.0,2008-01-03,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928366,2008,12,13,6,12:50:00,12:20:00,16:17:00,15:52:00,DL,1621,N938DL,147.0,152.0,120.0,25.0,30.0,MSP,ATL,906,9.0,18.0,3.0,0.0,0.0,0.0,22.0,2008-12-13,25.0
1928367,2008,12,13,6,06:57:00,06:00:00,09:04:00,07:49:00,DL,1631,N3743H,127.0,109.0,78.0,75.0,57.0,RIC,ATL,481,15.0,34.0,0.0,57.0,18.0,0.0,0.0,2008-12-13,75.0
1928368,2008,12,13,6,10:07:00,08:47:00,11:49:00,10:10:00,DL,1631,N909DA,162.0,143.0,122.0,99.0,80.0,ATL,IAH,689,8.0,32.0,1.0,0.0,19.0,0.0,79.0,2008-12-13,99.0
1928369,2008,12,13,6,12:51:00,12:40:00,14:46:00,14:37:00,DL,1639,N646DL,115.0,117.0,89.0,9.0,11.0,IAD,ATL,533,13.0,13.0,0.0,0.0,0.0,0.0,0.0,2008-12-13,0.0


In [24]:
# Reordering columns for better understanding
new_order = [
    'Date', 'year', 'month', 'day', 'DayOfWeek',
    # Flight Info
    'UniqueCarrier', 'FlightNum', 'TailNum',
    # Route
    'Origin', 'Dest', 'Distance',
    # Time
    'Scheduled_DepTime', 'Actual_DepTime','Departure_Delay',
    'Scheduled_ArrTime', 'Actual_ArrTime','Arrival_Delay',
    # Delay
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay','Delay_sum',
    # Duration
    'ActualElapsedTime', 'Expected_ElapsedTime', 'AirTime',
    'TaxiIn', 'TaxiOut'
]

Delayed_Flights=Delayed_Flights[new_order]
Delayed_Flights.head(10)

,Date,year,month,day,DayOfWeek,UniqueCarrier,FlightNum,TailNum,Origin,Dest,Distance,Scheduled_DepTime,Actual_DepTime,Departure_Delay,Scheduled_ArrTime,Actual_ArrTime,Arrival_Delay,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,Delay_sum,ActualElapsedTime,Expected_ElapsedTime,AirTime,TaxiIn,TaxiOut
0,2008-01-03,2008,1,3,4,WN,335,N712SW,IAD,TPA,810,19:55:00,20:03:00,8.0,22:25:00,22:11:00,-14.0,0.0,0.0,0.0,0.0,0.0,0.0,128.0,150.0,116.0,4.0,8.0
1,2008-01-03,2008,1,3,4,WN,3231,N772SW,IAD,TPA,810,07:35:00,07:54:00,19.0,10:00:00,10:02:00,2.0,0.0,0.0,0.0,0.0,0.0,0.0,128.0,145.0,113.0,5.0,10.0
2,2008-01-03,2008,1,3,4,WN,448,N428WN,IND,BWI,515,06:20:00,06:28:00,8.0,07:50:00,08:04:00,14.0,0.0,0.0,0.0,0.0,0.0,0.0,96.0,90.0,76.0,3.0,17.0
3,2008-01-03,2008,1,3,4,WN,3920,N464WN,IND,BWI,515,17:55:00,18:29:00,34.0,19:25:00,19:59:00,34.0,2.0,0.0,0.0,0.0,32.0,34.0,90.0,90.0,77.0,3.0,10.0
4,2008-01-03,2008,1,3,4,WN,378,N726SW,IND,JAX,688,19:15:00,19:40:00,25.0,21:10:00,21:21:00,11.0,0.0,0.0,0.0,0.0,0.0,0.0,101.0,115.0,87.0,4.0,10.0
5,2008-01-03,2008,1,3,4,WN,509,N763SW,IND,LAS,1591,18:30:00,19:37:00,67.0,19:40:00,20:37:00,57.0,10.0,0.0,0.0,0.0,47.0,57.0,240.0,250.0,230.0,3.0,7.0
6,2008-01-03,2008,1,3,4,WN,100,N690SW,IND,MCO,828,07:00:00,07:06:00,6.0,09:15:00,09:16:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,130.0,135.0,106.0,5.0,19.0
7,2008-01-03,2008,1,3,4,WN,1333,N334SW,IND,MCO,828,15:10:00,16:44:00,94.0,17:25:00,18:45:00,80.0,8.0,0.0,0.0,0.0,72.0,80.0,121.0,135.0,107.0,6.0,8.0
8,2008-01-03,2008,1,3,4,WN,2272,N263WN,IND,MDW,162,10:20:00,10:29:00,9.0,10:10:00,10:21:00,11.0,0.0,0.0,0.0,0.0,0.0,0.0,52.0,50.0,37.0,6.0,9.0
9,2008-01-03,2008,1,3,4,WN,675,N286WN,IND,PHX,1489,14:25:00,14:52:00,27.0,16:25:00,16:40:00,15.0,3.0,0.0,0.0,0.0,12.0,15.0,228.0,240.0,213.0,7.0,8.0


In [25]:
Delayed_Flights.to_csv('Cleaned_delayed_flights.csv',index=False)
